In [1]:
%load_ext autoreload
%autoreload 2

In [24]:
from pathlib import Path
import random
from text.extract import get_content
from text.clean import clean_minutes
from text.corpus import save_docs, load_docs
from text.corpus import save_records, load_records
from text.segment import to_records
from text.split import assign_splits, split_summary #, sample_split 


In [3]:
MINUTES = Path(r"C:\DS_ML\machine-learning-engineer\inflation-intelligence\data\raw\feasibility_2026-09-10\rpm_minutes")
COMUNICADOS = Path(r"C:\DS_ML\machine-learning-engineer\inflation-intelligence\data\chile_banco_central")
INTERIM = Path(r"C:\DS_ML\machine-learning-engineer\inflation-intelligence\data\interim")

In [4]:
minutes = get_content(MINUTES, "minutes")
for doc_id, doc in minutes.items():
    cleaned, report = clean_minutes(doc["text"])
    doc["cleaned"] = cleaned
    doc["report"] = report

bad = [d for d, doc in minutes.items() if doc["report"]["garbled_lines"] > 0]
print(len(minutes), "documents; garbled:", bad)

Ignoring wrong pointing object 136 0 (offset 0)
Ignoring wrong pointing object 145 0 (offset 0)
Ignoring wrong pointing object 157 0 (offset 0)
Ignoring wrong pointing object 133 0 (offset 0)
Ignoring wrong pointing object 143 0 (offset 0)


94 documents; garbled: ['minutes_2024_04_02', 'minutes_2024_01_31']


In [5]:
garbled = [doc_id for doc_id, doc in minutes.items() if doc["report"]["garbled_lines"] > 0]
minutes = {doc_id: doc for doc_id, doc in minutes.items() if doc_id not in garbled}

print("dropped:", garbled)        # expect minutes_2024_01_31, minutes_2024_04_02
print(len(minutes), "documents kept")

dropped: ['minutes_2024_04_02', 'minutes_2024_01_31']
92 documents kept


In [6]:
path = INTERIM / "minutes_clean.jsonl"
# save_docs(minutes, path)
minutes = load_docs(path)

In [7]:
records = to_records(minutes)

records

[{'doc_id': 'minutes_2024_05_23',
  'doc_type': 'minutes',
  'meeting_date': '2024-05-23',
  'sent_idx': 0,
  'sentence': 'En el ámbito externo, la inflación mundial había seguido disminuyendo, aunque a un ritmo más moderado, debido a la elevada persistencia de los componentes de servicios.',
  'cluster': None,
  'split': None,
  'stance': None,
  'sentiment': None},
 {'doc_id': 'minutes_2024_05_23',
  'doc_type': 'minutes',
  'meeting_date': '2024-05-23',
  'sent_idx': 1,
  'sentence': 'En Estados Unidos de América, tras sucesivas sorpresas al alza en la inflación durante el primer trimestre del año, el dato de abril había mostrado una desaceleración del componente subyacente.',
  'cluster': None,
  'split': None,
  'stance': None,
  'sentiment': None},
 {'doc_id': 'minutes_2024_05_23',
  'doc_type': 'minutes',
  'meeting_date': '2024-05-23',
  'sent_idx': 2,
  'sentence': 'En la Eurozona, la inflación continuaba moderándose, favorecida principalmente por los bienes, y mantenía un len

In [ ]:
from collections import Counter, defaultdict
from text.dedup import collect_names, build_name_pattern, assign_clusters, merge_near_duplicates
from text.corpus import save_records

# 1. Cluster: exact match first, then merge near-duplicates
names = build_name_pattern(collect_names(r["sentence"] for r in records))
records = assign_clusters(records, names)
records, merges = merge_near_duplicates(records, names, threshold=0.85)

# 2. The merges the Jaccard pass made, lowest scores first (the riskiest ones)
print(len(merges), "merges")
for score, a, b in sorted(merges)[:15]:
    print(score, "|", a[:70], "|", b[:70])

# 3. Final clusters: sizes, using the cluster numbers themselves
members = defaultdict(list)
for r in records:
    members[r["cluster"]].append(r)

sizes = Counter({cluster: len(rs) for cluster, rs in members.items()})
shared = sum(n for n in sizes.values() if n > 1)
print(f"\n{len(members)} clusters; {shared} sentences share a cluster")

# 4. The largest clusters: two different original sentences from each, with dates
for cluster, n in sizes.most_common(10):
    distinct = list({r["sentence"]: r for r in members[cluster]}.values())
    print(f"\n--- cluster {cluster}: {n} sentences, {len(distinct)} distinct")
    for r in distinct[:2]:
        print(" ", r["meeting_date"], r["sentence"][:110])

# 5. Save
save_records(records, INTERIM / "minutes_sentences.jsonl")

37 merges
0.85 | # a años de dos económicas eee en encuesta eof expectativas financiero | # a años de dos económicas eee en encuesta eof expectativas financiero
0.85 | # acordó con consejeros consejo de del el en la los mantener president | # acordó anterior con consejeros consejo de del el en la lo los manten
0.85 | adecuadas coincidieron consejeros de decisión en eran la las los monet | adecuadas coincidieron consejeros de decisión división en eran estudio
0.86 | # a años de dos económicas eee en encuesta eof expectativas financiero | # a años de dos económicas eee en encuesta eof expectativas financiero
0.86 | # aumentar consejeros en hasta la los pb por presidente tpm votaron y | # aumentar consejeros en hasta la los pb por tpm vicepresidente votaro
0.86 | # consejeros en hasta la los pb por presidente reducir tpm votaron y | # aumentar consejeros en hasta la los pb por presidente tpm votaron y
0.86 | # consejeros en hasta la los pb por reducir tpm votaron y | # consejeros en hasta

In [17]:
from collections import Counter

per_meeting = Counter(r["meeting_date"] for r in records)
total = 0
for date in sorted(per_meeting, reverse=True):
        total += per_meeting[date]
        print(date, per_meeting[date], total)
        if total > 700:
            break

2026-07-28 65 65
2026-06-16 76 141
2026-04-28 52 193
2026-03-24 65 258
2026-01-27 63 321
2025-12-16 55 376
2025-10-28 64 440
2025-09-09 83 523
2025-07-29 60 583
2025-06-17 70 653
2025-04-29 75 728


In [25]:
### 
records = assign_splits(records=records, test_from='2025-07-29')
# summary = split_summary(records)
# for split in summary:
#     print(split, summary[split]['sentences'])
def add_context(records):
    """Store each sentence's previous sentence, for the labeling CLI."""
    by_key = {(r["doc_id"], r["sent_idx"]): r["sentence"] for r in records}
    for r in records:
        r["context"] = by_key.get((r["doc_id"], r["sent_idx"] - 1))
    return records

def sample_split(records, split, n, seed, periods=None):

    rng = random.Random(seed)
    by_cluster = defaultdict(list)

    for r in records:
        if r['split'] == split:
            by_cluster[r["cluster"]].append(r)

    pool = [rng.choice(group) for group in by_cluster.values()]

    if periods is None:
        return rng.sample(pool, min(n, len(pool)))

        
    per_period = n // len(periods)
    sample = []
    for start, end in periods:                           # e.g. ("2016", "2019")
        in_period = [r for r in pool if start <= r["meeting_date"][:4] <= end]
        sample += rng.sample(in_period, min(per_period, len(in_period)))
    return sample
        
records = add_context(records)

PERIODS = [("2016", "2019"), ("2020", "2021"), ("2022", "2023"), ("2024", "2025")]
test_sample  = sample_split(records, "test",  400, seed=42)
dev_sample   = sample_split(records, "dev",   200, seed=42, periods=PERIODS)
train_sample = sample_split(records, "train", 400, seed=42, periods=PERIODS)

print(len(test_sample), len(dev_sample), len(train_sample))        


400 150 400


In [ ]:
records = add_context(records)

PERIODS = [("2016", "2019"), ("2020", "2021"), ("2022", "2023"), ("2024", "2025")]
test_sample  = sample_split(records, "test",  400, seed=42)
dev_sample   = sample_split(records, "dev",   200, seed=42, periods=PERIODS)
train_sample = sample_split(records, "train", 400, seed=42, periods=PERIODS)

print(len(test_sample), len(dev_sample), len(train_sample))

In [26]:
all_clusters = [r["cluster"] for s in (test_sample, dev_sample, train_sample) for r in s]
print("duplicate clusters:", len(all_clusters) - len(set(all_clusters)))   # expect 0
print("excluded:", sum(r["split"] == "excluded" for s in (test_sample, dev_sample, train_sample) for r in s))  # expect 0

duplicate clusters: 0
excluded: 0


In [27]:
save_records(test_sample,  INTERIM / "sample_test.jsonl")
save_records(dev_sample,   INTERIM / "sample_dev.jsonl")
save_records(train_sample, INTERIM / "sample_train.jsonl")